In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. 샌드위치 분산 및 복합표본 회귀 함수 정의
def _vcl(U, strata, psu):
    t = pd.DataFrame(U); t['st'] = strata; t['cl'] = psu
    uc = t.groupby(['st', 'cl']).sum(numeric_only=True)
    p = U.shape[1]
    B = np.zeros((p, p))
    for s, g in uc.groupby(level=0):
        m = len(g)
        if m > 1:
            mat = g.to_numpy()
            D = mat - mat.mean(axis=0)
            B += (m / (m - 1)) * (D.T @ D)
    return B

def svyglm(y, X, w, strata, psu, family='binomial'):
    y = np.asarray(y, float); X = np.asarray(X, float); w = np.asarray(w, float)
    if family == 'binomial':
        m = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues; v = mu * (1 - mu)
    else:
        m = sm.GLM(y, X, family=sm.families.Gaussian(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues; v = np.ones_like(mu)
    b = m.params
    v_col = w * v
    A = X.T @ (v_col[:, None] * X)
    U = (w * (y - mu))[:, None] * X
    B = _vcl(U, strata, psu)
    Ai = np.linalg.inv(A)
    V = Ai @ B @ Ai
    return b, np.sqrt(np.diag(V)), m

def or_ci(b, se, i=1):
    return np.exp(b[i]), np.exp(b[i] - 1.96 * se[i]), np.exp(b[i] + 1.96 * se[i])

# 2. 데이터 및 복합표본 설계 변수 불러오기
try:
    w = df['wt'].to_numpy(float)
    ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()
except NameError:
    DATA = 'knhanes_like_week4.csv'
    df = pd.read_csv(DATA)
    w = df['wt'].to_numpy(float)
    ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

# 3. 필요한 파생변수 생성 (male, past_smoker 등)
if 'male' not in df.columns:
    df['male'] = (df['sex'] == 1).astype(float)

if 'past_smoker' not in df.columns:
    df['past_smoker'] = (df['smk'] == 2).astype(float)

# 4. B-1 보정집합 변수 설정 (연령, 남성, 소득, 교육수준)[cite: 2]
B1_VARS = ['age', 'male', 'income', 'edu']

# 5. Model 1 (무보정 모형) 분석[cite: 1]
X_m1 = sm.add_constant(df[['past_smoker']].astype(float))
b_m1, se_m1, fit_m1 = svyglm(df['htn'], X_m1, w, ST, PS)
or_m1, lo_m1, hi_m1 = or_ci(b_m1, se_m1, i=1)

# 6. Model 2 (보정 모형) 분석[cite: 1]
X_m2 = sm.add_constant(df[['past_smoker'] + B1_VARS].astype(float))
b_m2, se_m2, fit_m2 = svyglm(df['htn'], X_m2, w, ST, PS)
or_m2, lo_m2, hi_m2 = or_ci(b_m2, se_m2, i=1)

# 7. B-2 결과 출력
print("=" * 65)
print(" [B-2 팀 과제 결과: 과거흡연 -> 고혈압(htn) 복합표본 로지스틱 회귀]")
print("=" * 65)
print(f"1) 무보정 모형 (Model 1): OR = {or_m1:.3f} (95% CI: {lo_m1:.3f} - {hi_m1:.3f})")
print(f"2) 보정 모형   (Model 2): OR = {or_m2:.3f} (95% CI: {lo_m2:.3f} - {hi_m2:.3f})")
print("=" * 65)
print(f"* 보정 변수: {', '.join(B1_VARS)}")
print("* 설계 기반(샌드위치) 표준오차 적용 완료")
print("=" * 65)

 [B-2 팀 과제 결과: 과거흡연 -> 고혈압(htn) 복합표본 로지스틱 회귀]
1) 무보정 모형 (Model 1): OR = 1.077 (95% CI: 0.911 - 1.274)
2) 보정 모형   (Model 2): OR = 1.009 (95% CI: 0.833 - 1.223)
* 보정 변수: age, male, income, edu
* 설계 기반(샌드위치) 표준오차 적용 완료
